Crossing Tagging - Sendim
=
Sendim version of *Tagueamento Cruzamentos 2*. It reads the crossing tags from an Excel export, sorts them into crossing categories, and writes AutoCAD scripts that re-insert them as multileaders on one layer per category.

**Differences from v2:** the input is an Excel file, and the tags are **not** renumbered. `sort_by_xy_coordinates` and `tagging` are defined but never called, so the tags are re-inserted with the text they already have.

**Workflow overview:**
1. Read the existing crossing tags and their positions.
2. Sort each tag into one of 7 crossing categories by the cable/interference codes in its text.
3. Write two AutoCAD scripts: one that creates a layer per category and one that re-inserts every tag as an `MLEADER` on its category layer.

**Tag format:** `CR.<system>.<interference>-NNN` (e.g. `CR.LVAC.LVDC-001`, `CR.MV.DRN-012`).

**Input prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `Sendim - interferências.xlsx` in `C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Cruzamentos\` (hardcoded). Only columns **C** (tag text), **M** (X) and **N** (Y) of the first sheet are read.
- It is read with `header=None`, so a header row is read as a data row. It is only dropped because its text doesn't contain `LVAC` or `MV`.
- Every cell of column C must be text: an empty cell (NaN) makes the `in` test raise a `TypeError`.
- The tag text is matched case-sensitively (`LVAC`, `LVDC`, `MV`, `DRN`).

**Outputs / AutoCAD:**
- `Sendim - create_layers.scr`: creates `Crossing-AC_DC`, `Crossing-AC_DRN`, ... These are **not** the layers the leaders script uses.
- `Sendim - reinsert_tags.scr`: sets each `EMF_Crossing_<category>` layer and draws the multileaders. The `EMF_Crossing_*` layers must already exist in the drawing (`-LAYER S` fails on a missing layer). It doesn't erase the old tags.

Classifying and writing the scripts
=
- Reads columns C, M and N of the Excel file as `TAG`, `Position X`, `Position Y` (the names are given in the order of the columns in the sheet).
- `categories` maps each category code to a description. Only the keys are used: they name the DataFrames and the layers.
- `allocate` appends one row to the category DataFrame, with `Layer` = `EMF_Crossing_<code>`. The tag is `i[0]` and the coordinates `i[1]`, `i[2]` here, because of the column order.
- **Classification** (substring match on the tag, case-sensitive, first match wins, so the order matters):
  1. `LVAC.LVDC` → `AC_DC`
  2. `LVAC.DRN` → `AC_DRN`
  3. `LVDC.DRN` → `DC_DRN`
  4. `MV.DRN` → `MV_DRN`
  5. `MV.LVAC` → `MV_AC`
  6. any other tag containing `LVAC` → `AC_SEC`
  7. any other tag containing `MV` → `MV_SEC`
  - A tag that matches none of these (e.g. `CR.LVDC.WALL-001`) is dropped silently.
- `Type of Crossing` = the tag without the `CR.` prefix and the `-NNN` suffix (e.g. `LVAC.INTAC`). It is computed for the two "secondary" categories but not used in this version.
- No renumbering: `sort_by_xy_coordinates` and `tagging` are left over from v2 and never called.
- **Leaders script:** all the crossings are sorted by layer. On each layer change it writes `-LAYER S <layer>` followed by a blank line, which ends the `-LAYER` command. The `current_layer is not None` check is always true (`current_layer` starts as `[]`), so the blank line is always written, which is what the command needs.
- Each crossing is one `MLEADER`: arrow at the crossing point, landing 10 units left and 5 up (`x-10, y+5`), text = the tag. The current multileader style is used.
- **Layers script:** `_-LAYER _M Crossing-<category>` for each of the 7 categories (Make creates the layer and sets it current). It is written after the leaders script, but in AutoCAD it must be run **first**.
- **Warning:** the leaders go on `EMF_Crossing_<category>`, but the layers script creates `Crossing-<category>`.
- `time`/`start` and `numpy` are not used.

In [12]:
import time

import pandas as pd
import numpy as np

start = time.time()

column_names = ["TAG", "Position X", "Position Y"]

df_TAGs = pd.read_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Cruzamentos\Sendim - interferências.xlsx",
                      header=None, names=column_names, usecols="C,M,N")

categories = {
    "AC_DC": "Crossing AC Cables with DC Cables",
    "AC_DRN": "Crossing AC Cables with Civil Drainage",
    "AC_SEC": "Crossing AC Cables with Secondary Interferences",
    "DC_DRN": "Crossing DC Cables with Civil Drainage",
    "MV_DRN": "Crossing MV Cables with Civil Drainage",
    "MV_AC": "Crossing MV Cables with AC Cables",
    "MV_SEC": "Crossing MV Cables with Secondary Interferences"
}

dfs = {
    name: pd.DataFrame(columns=["Position X", "Position Y", "TAG", "Layer"])
    for name in categories.keys()
}


def allocate(df_name, i):
    df = dfs[df_name]
    new_row = {"Position X": i[1], "Position Y": i[2], "TAG": i[0], "Layer": ("EMF_Crossing_"+df_name)}
    df.loc[len(df)] = new_row


def sort_by_xy_coordinates(df):
    return df.sort_values(by=["Position Y", "Position X"], ascending=[False, True]).reset_index(drop=True)


def tagging(df):
    return df["TAG"].astype(str).str[:-3] + (df.index+1).astype(str).str.zfill(3)


for item in df_TAGs.values:
    if "LVAC.LVDC" in item[0]:
        allocate("AC_DC", item)
        continue
    elif "LVAC.DRN" in item[0]:
        allocate("AC_DRN", item)
        continue
    elif "LVDC.DRN" in item[0]:
        allocate("DC_DRN", item)
        continue
    elif "MV.DRN" in item[0]:
        allocate("MV_DRN", item)
        continue
    elif "MV.LVAC" in item[0]:
        allocate("MV_AC", item)
        continue
    elif "LVAC" in item[0]:
        allocate("AC_SEC", item)
        continue
    elif "MV" in item[0]:
        allocate("MV_SEC", item)

dfs["MV_SEC"]["Type of Crossing"] = dfs["MV_SEC"]["TAG"].astype(str).str[3:-4]
dfs["AC_SEC"]["Type of Crossing"] = dfs["AC_SEC"]["TAG"].astype(str).str[3:-4]



print(dfs)

scr_filename = r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Cruzamentos\Sendim - reinsert_tags.scr"

with open(scr_filename, 'w') as scr_file:
    current_layer = []
    all_leaders = []
    for df in dfs.values():
        for _, row in df.iterrows():
            all_leaders.append((
                row["Layer"],
                row["Position X"],
                row["Position Y"],
                row["TAG"]
            ))

    # Process leaders with correct command spacing
    for layer, x, y, tag in sorted(all_leaders, key=lambda x: x[0]):
        if layer != current_layer:
            scr_file.write(f"-LAYER S {layer}\n")
            if current_layer is not None:  # Only add newline if not first command
                scr_file.write("\n")
            current_layer = layer

        scr_file.write("MLEADER\n")
        scr_file.write(f"{x},{y}\n")
        scr_file.write(f"{x-10},{y+5}\n")
        scr_file.write(f"{tag}\n")
        # No extra newline here - critical!

    scr_file.write("\n")  # Final newline only at end

scr_filename = r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Cruzamentos\Sendim - create_layers.scr"

with open(scr_filename, 'w') as scr_file:
    for layer_name in categories.keys():
        scr_file.write(f"_-LAYER _M Crossing-{layer_name} \n")

{'AC_DC':      Position X   Position Y               TAG               Layer
0    49827.3743  156565.9116  CR.LVAC.LVDC-001  EMF_Crossing_AC_DC
1    49165.3694  156557.9101  CR.LVAC.LVDC-002  EMF_Crossing_AC_DC
2    49165.3575  156527.8960  CR.LVAC.LVDC-003  EMF_Crossing_AC_DC
3    49827.4188  156520.9116  CR.LVAC.LVDC-004  EMF_Crossing_AC_DC
4    49165.3119  156505.4235  CR.LVAC.LVDC-005  EMF_Crossing_AC_DC
..          ...          ...               ...                 ...
182  49649.9812  154791.3451  CR.LVAC.LVDC-183  EMF_Crossing_AC_DC
183  49794.2082  154768.8478  CR.LVAC.LVDC-184  EMF_Crossing_AC_DC
184  49650.0079  154761.3254  CR.LVAC.LVDC-185  EMF_Crossing_AC_DC
185  49497.5673  154748.6516  CR.LVAC.LVDC-186  EMF_Crossing_AC_DC
186  49650.0227  154746.3645  CR.LVAC.LVDC-187  EMF_Crossing_AC_DC

[187 rows x 4 columns], 'AC_DRN':     Position X   Position Y              TAG                Layer
0   49831.9670  156534.9140  CR.LVAC.DRN-001  EMF_Crossing_AC_DRN
1   49773.6051  156

Empty cell (unused).